# **Cross-Family LLM Stitching: Comparing Reconstruction and Prediction-Based Objectives**

*COMS 4705 - Natural Language Processing - Final Project*

### **Authors**

Caroline Zhang (cz2977@columbia.edu)

Siddharth Grover (sg4792@columbia.edu)

Edwin Argudo (eaa2193@columbia.edu)

### **GitHub Repo**

https://github.com/edwin-a14/coms-4705-project-cross-model-stitching

## 1. Install Dependencies

In [1]:

%pip install -q \
    "transformers>=4.45,<5" \
    "accelerate>=1.0" \
    "datasets>=3,<5" \
    "huggingface_hub>=0.34,<1.0" \
    safetensors

## 2. Connect to Google Drive

In order to connect to the shared Google Drive folder, you need to create a shortcut from the shared folder into your personal My Drive. Do the following:
1. Go to drive.google.com
2. Locate the shared `coms-4705-project` folder, click inside
3. Click on the down-arrow next to the folder name
4. **Organize > Add shortcut > All Locations > My Drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

os.chdir('/content/drive/MyDrive/coms-4705-project')
print(f'Working directory set to: {os.getcwd()}')

files = os.listdir('.')
print(f'Read access confirmed, found {len(files)} items')

Working directory set to: /content/drive/MyDrive/coms-4705-project
Read access confirmed, found 0 items


## 3. Project Configuration

`meta-llama/Llama-3.2-3B` requires us to accept license before being able to access it on HuggingFace, so you'll need to go to https://huggingface.co/meta-llama/Llama-3.2-3B and fill out the "Request Access" form with your personal information.

In [4]:
MODELS = {
    "llama": "meta-llama/Llama-3.2-3B",  # NOTE: this model requires accepted license/access
    "qwen":  "Qwen/Qwen2.5-3B"
}

## 4. Verify HuggingFace access

To connect HuggingFace to this notebook:

1. Go to https://huggingface.co/settings/tokens
2. Create an Access Token with write privileges
3. In this notebook, go to the **Secrets** tab, and add a secret with name: `HF_TOKEN`, value: `<Access Token>`, and turn on notebook access
4. Restart the runtime, and run the cells below to verify access

In [5]:
from google.colab import userdata
from huggingface_hub import login, whoami, hf_hub_download
from huggingface_hub.utils import GatedRepoError

try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)

    user = whoami()
    print(f"Authenticated as: {user['name']}")

    config_path = hf_hub_download(
        repo_id=MODELS["llama"],
        filename="config.json"
    )
    print(f"Access confirmed for {MODELS["llama"]}!")

except userdata.SecretNotFoundError:
    print("Secret 'HF_TOKEN' not found. Add it in the Colab Secrets sidebar.")
except userdata.NotebookAccessError:
    print("Notebook access disabled. Toggle on 'Notebook access' for HF_TOKEN.")
except GatedRepoError:
    print(f"Access denied for {MODELS["llama"]}. Accept the license on its Hugging Face page.")

Authenticated as: edlose
Access confirmed for meta-llama/Llama-3.2-3B!


## 5. Verify Model Configuration and Tokenization

In [11]:

from transformers import AutoConfig, AutoTokenizer

qwen_id = MODELS["qwen"]

config = AutoConfig.from_pretrained(qwen_id)
tokenizer = AutoTokenizer.from_pretrained(qwen_id)

print("Model:", qwen_id)
print("Hidden size:", config.hidden_size)
print("Number of layers:", config.num_hidden_layers)
print("Attention heads:", config.num_attention_heads)
print("KV heads:", config.num_key_value_heads)
print("Vocabulary size:", config.vocab_size)

prompt = "The capital of France is"

tokens = tokenizer.tokenize(prompt)

print("\nPrompt:", prompt)
print("Tokens:", tokens)
print("Token count:", len(tokens))


Model: Qwen/Qwen2.5-3B
Hidden size: 2048
Number of layers: 36
Attention heads: 16
KV heads: 2
Vocabulary size: 151936

Prompt: The capital of France is
Tokens: ['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis']
Token count: 5


In [12]:

from transformers import AutoConfig, AutoTokenizer

llama_id = MODELS["llama"]

config = AutoConfig.from_pretrained(llama_id)
llama_tokenizer = AutoTokenizer.from_pretrained(llama_id)

print("Model:", llama_id)
print("Hidden size:", config.hidden_size)
print("Number of layers:", config.num_hidden_layers)
print("Attention heads:", config.num_attention_heads)
print("KV heads:", config.num_key_value_heads)
print("Vocabulary size:", config.vocab_size)

prompt = "The capital of France is"
tokens = llama_tokenizer.tokenize(prompt)

print("\nPrompt:", prompt)
print("Tokens:", tokens)
print("Token count:", len(tokens))

Model: meta-llama/Llama-3.2-3B
Hidden size: 3072
Number of layers: 28
Attention heads: 24
KV heads: 8
Vocabulary size: 128256

Prompt: The capital of France is
Tokens: ['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis']
Token count: 5


## 6. Load LAMBADA Dataset

In [8]:
from datasets import load_dataset

dataset = load_dataset(
    "cimec/lambada",
    split="validation",
    streaming=True
)

README.md: 0.00B [00:00, ?B/s]

In [9]:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODELS["qwen"])

# Get one LAMBADA example
example = next(iter(dataset))
text = example["text"].rstrip()

# Separate the final word from the context
prefix, target = text.rsplit(" ", 1)

# Tokenize both
prefix_ids = tokenizer.encode(prefix)
full_ids = tokenizer.encode(text)

# Check whether the token boundary is consistent
valid = full_ids[:len(prefix_ids)] == prefix_ids

print("Prefix:", prefix[-100:])
print("Target word:", target)
print("Valid token boundary:", valid)

if valid:
    next_token_id = full_ids[len(prefix_ids)]
    print("Next token:", repr(tokenizer.decode([next_token_id])))


Prefix: . brian and max were a lot of fun and always made her laugh as they hoisted her up to the top of the
Target word: cake
Valid token boundary: True
Next token: ' cake'
